In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score
import mlflow
from mlflow.models import infer_signature
from mlflow.models.signature import ModelSignature
from mlflow.types import Schema, ColSpec
from pathlib import Path
import re
import json

c:\Users\yeech\Documents\Project\churn_detection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
root_dir = Path.cwd().parent
dataset_dir = root_dir / "data" / "processed"
versions = set([v.name for v in dataset_dir.iterdir() if re.match(r"v\d+", v.name)])
version = sorted(versions)[-1] if versions else None
dataset_dir = dataset_dir / version
dataset_name = 'telco-customer-churn'

trainset = pd.read_csv(dataset_dir / "train.csv")
X_train = trainset.drop("Churn", axis=1)
Y_train = trainset["Churn"]

testset = pd.read_csv(dataset_dir / "test.csv")
X_test = testset.drop("Churn", axis=1)
Y_test = testset["Churn"]

schema = json.load(open(dataset_dir / "schema.json", "r"))

In [ ]:
def find_best_threshold(y_true, y_proba):
    best_t = 0
    best_f1 = 0

    for t in np.arange(0.1, 0.9, 0.01):
        y_pred = (y_proba >= t).astype(int)
        f1 = f1_score(y_true, y_pred)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    return best_t, best_f1

def build_fit_pipeline(schema, model,Y_train, X_train):
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), schema["features"]["numerical"]),
            ("cat", OneHotEncoder(handle_unknown="ignore"), schema["features"]["categorical"]),
        ]
    )
    
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    pipeline.fit(X_train, Y_train)
        
    return pipeline

def evaluate_model(model, Y_test, X_test):
    classes = model.classes_
    pos_index = list(classes).index("Yes")
    
    y_proba = model.predict_proba(X_test)[:, pos_index]
    
    Y_test_binary = (Y_test == "Yes").astype(int)
    best_t, _ = find_best_threshold(Y_test_binary, y_proba)
    ypred = (y_proba >= best_t).astype(int)
    
    print(classification_report(Y_test_binary, ypred))
    class_report_dict = classification_report(Y_test_binary, ypred, output_dict=True)
    return class_report_dict , best_t 

def mlflow_log(model,Y_train, X_train, parameters, mlflow_experiment, class_report_dict, version, dataset_name):
    
    sign = infer_signature(X_train,model.predict(X_train))
    
    if sign.outputs.inputs[0].type == 'object':
        new_output = Schema([ColSpec("string")])

        signature = ModelSignature(
            inputs=sign.inputs,
            outputs=new_output
        )
    else:
        signature = sign
    train_ds = mlflow.data.from_pandas(
        X_train.assign(Churn=Y_train),
        name=dataset_name + ':' + version
    )
    
    with mlflow.start_run(experiment_id=mlflow_experiment.experiment_id):
        mlflow.log_params(parameters)
        mlflow.log_param("data_version", version)
        
        mlflow.log_metric("accuracy", class_report_dict["accuracy"])
        mlflow.log_metric("precision", class_report_dict["1"]["precision"])
        mlflow.log_metric("recall", class_report_dict["1"]["recall"])
        mlflow.log_metric("f1-score", class_report_dict["1"]["f1-score"])
        
        mlflow.log_input(train_ds, context="training")
        
        mlflow.sklearn.log_model(model, parameters['model'], signature=signature)
    
def ml_pipeline(model, Y_train, X_train, X_test, Y_test, parameters, mlflow_experiment, version, dataset_name):
    parameters['model'] = model.__class__.__name__
    model = build_fit_pipeline(schema, model, Y_train, X_train)
    class_report_dict , parameters['best_threshold'] = evaluate_model(model, Y_test, X_test)
    mlflow_log(model, Y_train, X_train, parameters, mlflow_experiment, class_report_dict, version, dataset_name)

In [4]:
mlflow.set_tracking_uri((root_dir / "mlruns").as_uri())
exp = mlflow.set_experiment("churn_detection")

try_parameters = {
    "n_estimators": [50, 100, 200,500],
    "class_weight": ["balanced", None],
    "random_state": 42
}

for n_est in try_parameters["n_estimators"]:
    for cw in try_parameters["class_weight"]:
        parameters = {
            "n_estimators": n_est,
            "class_weight": cw,
            "random_state": try_parameters["random_state"]
        }
        model = RandomForestClassifier(**parameters)
        ml_pipeline(model, Y_train, X_train, X_test, Y_test, parameters, exp, version, dataset_name)

c:\Users\yeech\Documents\Project\churn_detection\.venv\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/04/24 19:24:51 INFO mlflow.tracking.fluent: Experiment with name 'churn_detection' does not exist. Creating a new experiment.


              precision    recall  f1-score   support

           0       0.91      0.68      0.78      1033
           1       0.48      0.81      0.60       374

    accuracy                           0.71      1407
   macro avg       0.69      0.74      0.69      1407
weighted avg       0.79      0.71      0.73      1407



c:\Users\yeech\Documents\Project\churn_detection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
c:\Users\yeech\Documents\Project\churn_detection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing v

              precision    recall  f1-score   support

           0       0.88      0.76      0.82      1033
           1       0.52      0.71      0.60       374

    accuracy                           0.75      1407
   macro avg       0.70      0.74      0.71      1407
weighted avg       0.78      0.75      0.76      1407



c:\Users\yeech\Documents\Project\churn_detection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
c:\Users\yeech\Documents\Project\churn_detection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing v

...

c:\Users\yeech\Documents\Project\churn_detection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
c:\Users\yeech\Documents\Project\churn_detection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing v

              precision    recall  f1-score   support

           0       0.89      0.74      0.81      1033
           1       0.51      0.74      0.60       374

    accuracy                           0.74      1407
   macro avg       0.70      0.74      0.70      1407
weighted avg       0.79      0.74      0.75      1407



c:\Users\yeech\Documents\Project\churn_detection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
c:\Users\yeech\Documents\Project\churn_detection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing v

              precision    recall  f1-score   support

           0       0.88      0.76      0.82      1033
           1       0.52      0.71      0.60       374

    accuracy                           0.75      1407
   macro avg       0.70      0.74      0.71      1407
weighted avg       0.79      0.75      0.76      1407



c:\Users\yeech\Documents\Project\churn_detection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
c:\Users\yeech\Documents\Project\churn_detection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing v

In [5]:
try_parameters = {
    'C': [0.1, 1, 10],
    'penalty': ['l1', 'l2']
}

for C in try_parameters['C']:
    for penalty in try_parameters['penalty']:
        if penalty == 'l1':
            solver = 'liblinear'
        else:
            solver = 'lbfgs'
        parameters = {
            'C': C,
            'penalty': penalty,
            'solver': solver,
            'max_iter': 1000
        }
        model = LogisticRegression(**parameters)
        ml_pipeline(model, Y_train, X_train, X_test, Y_test, parameters, exp, version, dataset_name)

c:\Users\yeech\Documents\Project\churn_detection\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\yeech\Documents\Project\churn_detection\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


              precision    recall  f1-score   support

           0       0.90      0.73      0.81      1033
           1       0.52      0.79      0.62       374

    accuracy                           0.75      1407
   macro avg       0.71      0.76      0.72      1407
weighted avg       0.80      0.75      0.76      1407



c:\Users\yeech\Documents\Project\churn_detection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
c:\Users\yeech\Documents\Project\churn_detection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing v

              precision    recall  f1-score   support

           0       0.90      0.74      0.81      1033
           1       0.52      0.78      0.62       374

    accuracy                           0.75      1407
   macro avg       0.71      0.76      0.72      1407
weighted avg       0.80      0.75      0.76      1407



c:\Users\yeech\Documents\Project\churn_detection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/04/24 19:25:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/24 19:25:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format r

...

c:\Users\yeech\Documents\Project\churn_detection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/04/24 19:25:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/24 19:25:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format r

              precision    recall  f1-score   support

           0       0.90      0.74      0.82      1033
           1       0.52      0.78      0.63       374

    accuracy                           0.75      1407
   macro avg       0.71      0.76      0.72      1407
weighted avg       0.80      0.75      0.76      1407



c:\Users\yeech\Documents\Project\churn_detection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
c:\Users\yeech\Documents\Project\churn_detection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing v

              precision    recall  f1-score   support

           0       0.90      0.76      0.82      1033
           1       0.53      0.76      0.63       374

    accuracy                           0.76      1407
   macro avg       0.71      0.76      0.72      1407
weighted avg       0.80      0.76      0.77      1407



c:\Users\yeech\Documents\Project\churn_detection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/04/24 19:25:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/24 19:25:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format r

In [6]:
def register_best_model(mlflow_experiment):
    all_runs_df = mlflow.search_runs(experiment_ids=[mlflow_experiment.experiment_id], order_by=["metrics.recall DESC"])
    best_model = all_runs_df.iloc[0]
    best_run_id = best_model['run_id']
    model_name = best_model['params.model']
    mlflow.register_model(
        model_uri=f"runs:/{best_run_id}/{model_name}", 
        name="customer-churn-model"
        )

In [7]:
register_best_model(exp)

c:\Users\yeech\Documents\Project\churn_detection\.venv\Lib\site-packages\mlflow\tracking\_model_registry\utils.py:220: FutureWarning: The filesystem model registry backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri)
Successfully registered model 'customer-churn-model'.
2026/04/24 19:25:32 WARNING mlflow.tracking._model_registry.fluent: Run with id 62bc5213c5ba419fbb12d1679e0904ee has no artifacts at artifact path 'RandomForestClassifier', registering model based on models:/m-d128d7f2ea104ac699ad52b4c1dd68dd instead
Created version '1' of model 'customer-churn-model'.
